In [ ]:
%pip install -q librosa numpy torch torchvision scikit-learn

from pathlib import Path
import json
import pandas as pd
import numpy as np
import librosa

PROJECT_ROOT = Path(r"C:\Users\Scowly\575_proj\cse575-project-repo")
DATA_ROOT = PROJECT_ROOT / "Data"
OUT_ROOT = PROJECT_ROOT / "processed_mels"
OUT_ROOT.mkdir(exist_ok=True)

print("PROJECT_ROOT =", PROJECT_ROOT)
print("DATA_ROOT =", DATA_ROOT)
print("OUT_ROOT =", OUT_ROOT)

In [ ]:
def parse_annotation_json(json_path, binary=True):
    """
    Returns a DataFrame with columns: start_sec, end_sec, label.
    Times are relative to record_start so you can index into audio.
    """
    json_path = Path(json_path)
    with open(json_path, "r") as f:
        data = json.load(f)

    record_start = float(data["record_start"])
    rows = []
    for ev in data["events"]:
        ev_type = ev["event_type"]           # "hypo" or "osa"
        ev_start_abs = float(ev["evnet_start"])  # note: 'evnet_start' in the JSON
        ev_dur = float(ev["event_duration"])

        start_rel = max(0.0, ev_start_abs - record_start)
        end_rel = start_rel + ev_dur

        if binary:
            label = "apnea"  # both hypo and osa considered apnea
        else:
            label = ev_type  # 'hypo' vs 'osa'

        rows.append({
            "start_sec": start_rel,
            "end_sec": end_rel,
            "label": label,
        })

    return pd.DataFrame(rows)

In [ ]:
def make_mel(
    y,
    sr,
    n_fft=1024,
    hop_length=256,
    n_mels=128,
    fmin=20,
    fmax=None,
):
    S = librosa.feature.melspectrogram(
        y=y,
        sr=sr,
        n_fft=n_fft,
        hop_length=hop_length,
        n_mels=n_mels,
        fmin=fmin,
        fmax=fmax,
    )
    S_db = librosa.power_to_db(S, ref=np.max)
    return S_db

def segment_to_mel(y, sr, start_sec, end_sec, **mel_kwargs):
    start_sample = int(start_sec * sr)
    end_sample = int(end_sec * sr)
    start_sample = max(0, start_sample)
    end_sample = min(len(y), end_sample)
    seg = y[start_sample:end_sample]
    if len(seg) == 0:
        return None
    return make_mel(seg, sr, **mel_kwargs)

def normalize_mel(mel):
    m_min = mel.min()
    m_max = mel.max()
    if m_max == m_min:
        return np.zeros_like(mel)
    return (mel - m_min) / (m_max - m_min)

In [ ]:
def preprocess_audio_segments_from_df(
    audio_path,
    segments_df,
    out_dir,
    sr=16000,
    n_fft=1024,
    hop_length=256,
    n_mels=128,
    fmin=20,
    fmax=None,
    min_duration_sec=5.0,
    prefix=None,
):
    audio_path = Path(audio_path)
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    # load and resample
    y, sr = librosa.load(audio_path.as_posix(), sr=sr)

    records = []
    base_prefix = prefix if prefix is not None else audio_path.stem

    for idx, row in segments_df.iterrows():
        start = float(row["start_sec"])
        end = float(row["end_sec"])
        label = row["label"]

        if end - start < min_duration_sec:
            continue

        mel = segment_to_mel(
            y, sr, start, end,
            n_fft=n_fft,
            hop_length=hop_length,
            n_mels=n_mels,
            fmin=fmin,
            fmax=fmax,
        )
        if mel is None:
            continue

        mel = normalize_mel(mel).astype(np.float32)

        mel_fname = f"{base_prefix}_seg_{idx:05d}.npy"
        mel_path = out_dir / mel_fname
        np.save(mel_path, mel)

        records.append({
            "file": str(mel_path),
            "label": label,
            "start_sec": start,
            "end_sec": end,
            "audio_file": str(audio_path),
        })

    if not records:
        return None

    meta_df = pd.DataFrame(records)
    meta_path = out_dir / f"{base_prefix}_metadata.csv"
    meta_df.to_csv(meta_path, index=False)
    print(f"[OK] {audio_path} → {len(records)} segments")
    return meta_path

In [ ]:
def preprocess_all_folders(
    data_root,
    out_root,
    sr=16000,
    n_fft=1024,
    hop_length=256,
    n_mels=128,
    fmin=20,
    fmax=None,
    min_duration_sec=5.0,
    binary_labels=True,
):
    data_root = Path(data_root)
    out_root = Path(out_root)
    seg_root = out_root / "mel_segments"
    seg_root.mkdir(parents=True, exist_ok=True)

    meta_paths = []

    for folder in sorted(p for p in data_root.iterdir() if p.is_dir()):
        # find annotation json
        ann_files = list(folder.glob("*annotation.json"))
        if not ann_files:
            print(f"[WARN] no annotation.json in {folder}")
            continue

        ann_path = ann_files[0]
        segments_df = parse_annotation_json(ann_path, binary=binary_labels)

        # all wavs in the folder
        wavs = sorted(folder.glob("*.wav"))
        if not wavs:
            print(f"[WARN] no wav files in {folder}")
            continue

        for wav_path in wavs:
            prefix = f"{folder.name}_{wav_path.stem}"
            meta_path = preprocess_audio_segments_from_df(
                audio_path=wav_path,
                segments_df=segments_df,
                out_dir=seg_root,
                sr=sr,
                n_fft=n_fft,
                hop_length=hop_length,
                n_mels=n_mels,
                fmin=fmin,
                fmax=fmax,
                min_duration_sec=min_duration_sec,
                prefix=prefix,
            )
            if meta_path is not None:
                meta_paths.append(meta_path)

    # combine all metadata
    if meta_paths:
        all_meta = pd.concat([pd.read_csv(p) for p in meta_paths], ignore_index=True)
        global_meta_path = out_root / "all_metadata.csv"
        all_meta.to_csv(global_meta_path, index=False)
        print(f"\n[GLOBAL] Combined metadata saved to {global_meta_path}")
    else:
        print("\n[GLOBAL] No segments created.")

In [ ]:
preprocess_all_folders(
    data_root=DATA_ROOT,
    out_root=OUT_ROOT,
    sr=16000,
    n_fft=1024,
    hop_length=256,
    n_mels=128,
    fmin=20,
    fmax=8000,
    min_duration_sec=5.0,
    binary_labels=False,
)

In [ ]:
from torch.utils.data import Dataset, DataLoader, Subset
import torch

class MelSegmentDataset(Dataset):
    def __init__(self, meta_csv, target_frames=256):
        self.df = pd.read_csv(meta_csv)
        self.paths = self.df["file"].tolist()
        self.labels = self.df["label"].tolist()

        classes = sorted(set(self.labels))
        self.class_to_idx = {c: i for i, c in enumerate(classes)}
        self.idx_to_class = {i: c for c, i in self.class_to_idx.items()}
        self.y = [self.class_to_idx[l] for l in self.labels]

        self.target_frames = target_frames

    def _pad_or_crop(self, mel):
        n_mels, T = mel.shape
        if T == self.target_frames:
            return mel
        if T > self.target_frames:
            start = (T - self.target_frames) // 2
            end = start + self.target_frames
            return mel[:, start:end]
        pad_width = self.target_frames - T
        return np.pad(mel, ((0, 0), (0, pad_width)), mode="constant")

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        mel = np.load(self.paths[idx])  # (n_mels, T)
        mel = self._pad_or_crop(mel)
        mel = mel[np.newaxis, :, :]     # (1, n_mels, T)
        mel = torch.from_numpy(mel).float()
        label = self.y[idx]
        return mel, label

dataset = MelSegmentDataset(OUT_ROOT / "all_metadata.csv", target_frames=256)

In [ ]:
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Subset

meta_csv = "processed_mels/all_metadata.csv"
full_df = pd.read_csv(meta_csv)
labels = full_df["label"].tolist()

indices = list(range(len(full_df)))
train_idx, temp_idx, _, temp_y = train_test_split(
    indices, labels, test_size=0.3, stratify=labels, random_state=42
)
val_idx, test_idx, _, _ = train_test_split(
    temp_idx, temp_y, test_size=0.5, stratify=temp_y, random_state=42
)

dataset = MelSegmentDataset(meta_csv, target_frames=256)

train_ds = Subset(dataset, train_idx)
val_ds   = Subset(dataset, val_idx)
test_ds  = Subset(dataset, test_idx)

batch_size = 32

train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=4)
val_loader   = DataLoader(val_ds,   batch_size=batch_size, shuffle=False, num_workers=4)
test_loader  = DataLoader(test_ds,  batch_size=batch_size, shuffle=False, num_workers=4)

In [ ]:
import torch
import torch.nn as nn
from torchvision import models

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# resnet18 with ImageNet weights (optional)
resnet = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

# change first conv to 1 channel
old_conv = resnet.conv1
resnet.conv1 = nn.Conv2d(
    in_channels=1,
    out_channels=old_conv.out_channels,
    kernel_size=old_conv.kernel_size,
    stride=old_conv.stride,
    padding=old_conv.padding,
    bias=old_conv.bias is not None
)

# if using pretrained weights, average RGB weights into single channel
with torch.no_grad():
    resnet.conv1.weight[:] = old_conv.weight.mean(dim=1, keepdim=True)

# change final FC to num_classes
num_classes = len(dataset.class_to_idx)
resnet.fc = nn.Linear(resnet.fc.in_features, num_classes)

resnet = resnet.to(device)

In [ ]:
import torch.optim as optim

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(resnet.parameters(), lr=1e-4)

def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for X, y in loader:
        X = X.to(device)  # (B, 1, n_mels, T)
        y = y.to(device)

        optimizer.zero_grad()
        outputs = model(X)           # (B, num_classes)
        loss = criterion(outputs, y)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * X.size(0)
        _, preds = outputs.max(1)
        correct += (preds == y).sum().item()
        total += y.size(0)

    return running_loss / total, correct / total

@torch.no_grad()
def eval_epoch(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0

    for X, y in loader:
        X = X.to(device)
        y = y.to(device)

        outputs = model(X)
        loss = criterion(outputs, y)

        running_loss += loss.item() * X.size(0)
        _, preds = outputs.max(1)
        correct += (preds == y).sum().item()
        total += y.size(0)

    return running_loss / total, correct / total

num_epochs = 10

for epoch in range(num_epochs):
    train_loss, train_acc = train_one_epoch(resnet, train_loader, optimizer, criterion, device)
    val_loss, val_acc     = eval_epoch(resnet, val_loader, criterion, device)
    print(f"Epoch {epoch+1}/{num_epochs} | "
          f"Train loss {train_loss:.4f} acc {train_acc:.3f} | "
          f"Val loss {val_loss:.4f} acc {val_acc:.3f}")

In [ ]:
@torch.no_grad()
def test_model(model, loader, device):
    model.eval()
    correct = 0
    total = 0
    all_preds = []
    all_true = []

    for X, y in loader:
        X = X.to(device)
        y = y.to(device)

        outputs = model(X)
        _, preds = outputs.max(1)

        all_preds.extend(preds.cpu().numpy().tolist())
        all_true.extend(y.cpu().numpy().tolist())
        correct += (preds == y).sum().item()
        total += y.size(0)

    acc = correct / total
    return acc, all_true, all_preds

test_acc, y_true, y_pred = test_model(resnet, test_loader, device)
print(f"Test accuracy: {test_acc:.3f}")